# Regression of Used Car Prices

**Curso:** Modelos y Simulación de Sistemas I  
**Universidad:** Universidad de Antioquia  
**Integrantes:** Juan Manuel Moscoso Torres, Slleider David Rojas Aleman y Samuel Castaño Arenas

Este proyecto aborda la estimación del precio de venta de autos usados. Es un problema supervisado de **regresión**; la variable objetivo es `price`. La fuente es la competencia [Regression of Used Car Prices — Kaggle Playground Series S4E9](https://www.kaggle.com/competitions/playground-series-s4e9).

### Índice
0. Título e introducción global  
1. Configuración  
2. Descripción del dataset  
3. Exploración de datos (EDA)  
4. Datos faltantes y estrategia  
5. Partición train/test  
6. Extracción y preprocesamiento  
7. Outliers y escala del objetivo  
8. Prevención de fuga de información  
9. Modelo base  
10. Modelo predictivo y validación cruzada  
11. Evaluación final en test  
12. Predicciones, residuos e importancia de variables  
13. Interpretación y siguientes versiones  
14. Guardado y prueba de inferencia

## 1. Configuración

La primera celda instala las dependencias fijadas en `requirements.txt` dentro del kernel activo. Después, esta sección importa las librerías, configura la semilla y las rutas, carga `features.py` y lee una sola vez el CSV. Para ejecutar el notebook completo desde el inicio, usa **Run All**: la instalación se ejecutará automáticamente antes de los imports. El resolvedor admite ejecución desde `fase-1/`, la raíz del proyecto o `fase-1/notebooks/`. Si la instalación actualizó paquetes ya cargados en el kernel, reinícialo y vuelve a ejecutar **Run All**.

In [ ]:
from pathlib import Path

current_dir = Path.cwd()
requirements_candidates = [
    current_dir / "requirements.txt",
    current_dir / "fase-1" / "requirements.txt",
    current_dir.parent / "requirements.txt",
]
requirements_path = next(
    (candidate for candidate in requirements_candidates if candidate.is_file()),
    None,
)

if requirements_path is None:
    raise FileNotFoundError(
        "No se encontró requirements.txt. Abre el notebook desde la raíz del proyecto, "
        "fase-1/ o fase-1/notebooks/."
    )

print(f"Instalando dependencias desde: {requirements_path}")
get_ipython().run_line_magic("pip", f'install --quiet -r "{requirements_path}"')
print("Dependencias listas.")

In [ ]:
from pathlib import Path
import platform
import sys

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer

RANDOM_STATE = 42

sns.set_theme(style="whitegrid", palette="colorblind")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.titleweight"] = "bold"

CURRENT_DIR = Path.cwd()
if (CURRENT_DIR / "features.py").exists():
    FASE_DIR = CURRENT_DIR
elif (CURRENT_DIR / "fase-1" / "features.py").exists():
    FASE_DIR = CURRENT_DIR / "fase-1"
elif (CURRENT_DIR.parent / "features.py").exists():
    FASE_DIR = CURRENT_DIR.parent
else:
    raise FileNotFoundError("No se encontró fase-1/features.py desde el directorio actual.")

if str(FASE_DIR) not in sys.path:
    sys.path.insert(0, str(FASE_DIR))

from features import (
    build_preprocessor,
    extract_engine_features,
    extract_transmission_features,
)

DATA_PATH = FASE_DIR / "data" / "train.csv"
df = pd.read_csv(DATA_PATH)

print(f"Python: {platform.python_version()}")
print(f"pandas: {pd.__version__}")
print(f"numpy: {np.__version__}")
print(f"scikit-learn: {sklearn.__version__}")

## 2. Descripción del dataset

Se inspeccionan la dimensión, una muestra y el tipo analítico de cada campo. El archivo ya fue cargado en la sección 1 y no se vuelve a leer.

### Inspección de la muestra

Se inspecciona la copia de trabajo cargada desde la ruta configurada en la sección 1. Primero se comprueba la dimensión, las columnas y una muestra de registros para confirmar que el archivo esperado se leyó correctamente; el CSV original permanece intacto.

In [ ]:
print(f"Dimensiones: {df.shape[0]:,} filas x {df.shape[1]} columnas")
display(df.head())

La lectura confirma que el archivo contiene **188,533 filas y 13 columnas**, con el objetivo `price` presente. La muestra sirve para inspeccionar el formato, pero no para inferir distribuciones; las conclusiones siguientes usan el conjunto completo.

### Tipo de variables

El siguiente resumen separa el tipo inferido por pandas del papel analítico de cada campo. `id` es un identificador aunque esté almacenado como entero; no representa una magnitud del vehículo.

In [ ]:
assert "price" in df.columns, "No se encontró la variable objetivo 'price'."

numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
categorical_cols = df.select_dtypes(exclude=np.number).columns.tolist()

roles = {}
for column in df.columns:
    if column == "id":
        roles[column] = "Identificador (descartar como predictor)"
    elif column == "price":
        roles[column] = "Objetivo numérico (regresión)"
    elif column in numeric_cols:
        roles[column] = "Predictora numérica"
    else:
        roles[column] = "Predictora categórica / texto"

schema = pd.DataFrame({
    "variable": df.columns,
    "tipo pandas": [str(df[column].dtype) for column in df.columns],
    "tipo analítico": [roles[column] for column in df.columns],
})
display(schema)
print(f"Predictores numéricos: { [c for c in numeric_cols if c not in ['id', 'price']] }")
print(f"Predictores categóricos/texto: {categorical_cols}")

`price` representa el precio observado del automóvil y es el objetivo numérico. Como limitaciones, los datos pueden reflejar la selección de anuncios disponibles en Kaggle, contener vehículos o versiones repetidos, y no incluyen necesariamente todos los factores del mercado ni una fecha de venta que permita validar cambios temporales. `id` identifica filas, no una característica del vehículo.

## 3. Exploración de datos (EDA)

El EDA es descriptivo y usa el conjunto completo; ninguna estadística de esta sección se usa para ajustar transformaciones ni el modelo. Todo lo aprendido de los datos (imputación, categorías, escalas, modelo) se ajusta solo con train.

Hay **dos variables numéricas predictoras** (`model_year` y `milage`), además de `id`, que solo identifica registros, y varias variables categóricas o de texto; `price` es el objetivo numérico. El precio observado va de autos económicos a vehículos de lujo, por lo que las medias y escalas pueden estar dominadas por valores extremos. El conjunto también puede reflejar sesgos de selección del mercado y categorías escritas de forma heterogénea.

### Valores faltantes

Se cuantifica la ausencia por columna, se visualiza su porcentaje y se compara el precio medio entre filas con y sin nulo para cada predictor afectado. Esa comparación detecta asociaciones descriptivas, no demuestra por sí sola que el mecanismo de ausencia no sea aleatorio ni una relación causal.

In [ ]:
missing_table = pd.DataFrame({
    "nulos": df.isna().sum(),
    "porcentaje": df.isna().mean().mul(100),
}).sort_values("porcentaje", ascending=False)
missing_display = missing_table.copy()
missing_display["porcentaje"] = missing_display["porcentaje"].map("{:.3f}%".format)
display(missing_display)

ax = missing_table["porcentaje"].sort_values().plot(kind="barh", color="#287271", figsize=(9, 6))
ax.set(title="Valores faltantes por variable", xlabel="Filas con valor faltante (%)", ylabel="Variable")
plt.tight_layout()
plt.show()

predictor_cols = [column for column in df.columns if column not in ["id", "price"]]
missing_in_range = missing_table.loc[
    missing_table.index.isin(predictor_cols)
    & missing_table["porcentaje"].between(0.1, 2.0)
].index.tolist()
print("Predictores con entre 0.1% y 2% de nulos:", missing_in_range or "Ninguno")

price_missing_rows = []
for column in predictor_cols:
    null_mask = df[column].isna()
    if null_mask.any():
        mean_without = df.loc[~null_mask, "price"].mean()
        mean_with = df.loc[null_mask, "price"].mean()
        price_missing_rows.append({
            "variable": column,
            "filas con nulo": int(null_mask.sum()),
            "precio medio sin nulo": mean_without,
            "precio medio con nulo": mean_with,
            "diferencia (con - sin)": mean_with - mean_without,
        })
price_by_missing = pd.DataFrame(price_missing_rows)
display(price_by_missing)

El mayor faltante está en `clean_title` (**11.36%**), seguido por `fuel_type` (**2.70%**) y `accident` (**1.30%**). La única predictora dentro del intervalo solicitado de **0.1% a 2%** es `accident`. En las tres variables, las filas con nulo muestran un precio medio superior al de las filas completas; por ejemplo, `clean_title` presenta aproximadamente **$63,568 frente a $41,354**. Esto sugiere que la ausencia puede contener señal o reflejar subgrupos distintos, pero la comparación marginal no prueba por sí sola el mecanismo de ausencia. En particular, `None reported` en `accident` es una categoría observada y no debe confundirse con un nulo.

### Estadísticas numéricas y valores atípicos

Se resumen `model_year`, `milage` y `price`, y se cuentan observaciones fuera de los límites de 1.5 veces el rango intercuartílico (IQR). Los boxplots ocultan los puntos extremos únicamente para que la caja sea legible; los registros permanecen en `df` y el conteo IQR los incluye.

In [ ]:
numeric_summary = df[["model_year", "milage", "price"]].describe().T
numeric_summary["skewness"] = df[["model_year", "milage", "price"]].skew()
display(numeric_summary)

iqr_rows = []
for column in ["model_year", "milage", "price"]:
    q1, q3 = df[column].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    count = int(((df[column] < lower) | (df[column] > upper)).sum())
    iqr_rows.append({
        "variable": column,
        "límite inferior": lower,
        "límite superior": upper,
        "observaciones fuera de IQR": count,
        "porcentaje fuera de IQR": count / len(df) * 100,
    })
iqr_table = pd.DataFrame(iqr_rows)
iqr_display = iqr_table.copy()
for column in ["límite inferior", "límite superior"]:
    iqr_display[column] = iqr_display[column].map("{:,.1f}".format)
iqr_display["porcentaje fuera de IQR"] = iqr_display["porcentaje fuera de IQR"].map("{:.2f}%".format)
display(iqr_display)

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
for axis, column in zip(axes, ["model_year", "milage", "price"]):
    sns.boxplot(y=df[column], ax=axis, color="#e9c46a", showfliers=False)
    axis.set_title(column)
    axis.set_ylabel("Valor")
fig.suptitle("Distribuciones numéricas (puntos extremos ocultos solo en el gráfico)")
plt.tight_layout()
plt.show()

El método IQR marca aproximadamente **5,132** observaciones de `model_year`, **1,766** de `milage` y **10,880** de `price` fuera de sus límites. Son candidatos para inspección, no necesariamente errores: en autos usados, antigüedad, kilometraje y precio extremos pueden ser legítimos y relevantes para el dominio.

### Distribución del objetivo

Se comparan el precio en escala original y `log1p(price)`. El sesgo y los histogramas permiten evaluar si una transformación del objetivo puede facilitar el ajuste de modelos sensibles a la asimetría; la decisión final debe considerar también la métrica de evaluación y la interpretación en dólares.

In [ ]:
price_skew = df["price"].skew()
log_price = np.log1p(df["price"])
log_price_skew = log_price.skew()
print(f"Sesgo de price: {price_skew:.3f}")
print(f"Sesgo de log1p(price): {log_price_skew:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df["price"], bins=60, ax=axes[0], color="#287271")
axes[0].set(title="Precio en escala original", xlabel="Precio", ylabel="Frecuencia")
sns.histplot(log_price, bins=60, ax=axes[1], color="#e76f51")
axes[1].set(title="Precio transformado con log1p", xlabel="log1p(price)", ylabel="Frecuencia")
plt.tight_layout()
plt.show()

`price` tiene un sesgo de **20.27**, mientras que `log1p(price)` queda cerca de simétrico (**0.108**). La transformación reduce claramente la cola derecha y conviene evaluarla como alternativa de objetivo en la validación posterior. Si se utiliza, las predicciones deben volver a la escala monetaria antes de compararlas con métricas expresadas en dólares, considerando que la inversión directa del log puede introducir sesgo.

### Variables categóricas

Se calcula la cardinalidad de las ocho variables categóricas solicitadas. Para las de cardinalidad alta se muestran las diez categorías más frecuentes; esto permite dimensionar la dispersión sin imprimir cientos o miles de niveles.

In [ ]:
categorical_columns = [
    "brand", "model", "fuel_type", "transmission",
    "ext_col", "int_col", "accident", "clean_title",
]
cardinality = pd.DataFrame({
    "variable": categorical_columns,
    "categorías distintas": [df[column].nunique(dropna=True) for column in categorical_columns],
    "valores faltantes": [int(df[column].isna().sum()) for column in categorical_columns],
}).sort_values("categorías distintas", ascending=False)
display(cardinality)

high_cardinality = cardinality.loc[
    cardinality["categorías distintas"] >= 50, "variable"
].tolist()
print("Top 10 categorías para variables con cardinalidad >= 50:")
for column in high_cardinality:
    top_categories = df[column].value_counts(dropna=False).head(10)
    print(f"\n{column} ({df[column].nunique(dropna=True)} categorías)")
    display(top_categories.rename_axis("categoría").to_frame("frecuencia"))

La cardinalidad observada varía desde **1 nivel** en `clean_title` hasta **1,897** en `model`; `brand`, `transmission`, `ext_col` e `int_col` también tienen al menos 50 niveles. `clean_title` solo contiene `Yes` entre sus valores observados, de modo que no permite comparar precios por categoría sin considerar sus nulos como un estado distinto. Entre los niveles más frecuentes destacan `F-150 XLT` en `model` (2,945 filas), `Black` en `ext_col` (48,658) e `int_col` (107,674), `Ford` en `brand` (23,088) y `A/T` en `transmission` (49,904), lo que muestra una concentración desigual de categorías.

Para visualizar asociaciones sin saturar los ejes, se grafican los niveles más frecuentes de cada variable con más de una categoría. Los boxplots usan escala logarítmica en el eje del precio y omiten puntos extremos solo en la figura; el precio original no se modifica.

In [ ]:
plot_columns = [
    "brand", "model", "fuel_type", "transmission", "ext_col", "int_col", "accident",
]
fig, axes = plt.subplots(3, 3, figsize=(18, 16))
for axis, column in zip(axes.flat, plot_columns):
    levels = df[column].value_counts(dropna=True).head(6).index.tolist()
    plot_data = df.loc[df[column].isin(levels), [column, "price"]]
    sns.boxplot(
        data=plot_data,
        x=column,
        y="price",
        order=levels,
        showfliers=False,
        color="#8ab17d",
        ax=axis,
    )
    axis.set_yscale("log")
    axis.set_title(f"price por {column} (6 niveles más frecuentes)")
    axis.set_xlabel("")
    axis.set_ylabel("Precio (escala log)")
    axis.tick_params(axis="x", rotation=35, labelsize=8)
for axis in axes.flat[len(plot_columns):]:
    axis.set_visible(False)
fig.suptitle("Precio por categorías frecuentes", y=1.01, fontsize=15)
plt.tight_layout()
plt.show()

Las distribuciones muestran diferencias relevantes entre categorías frecuentes, aunque con solapamiento y sin controlar por año, kilometraje, marca u otras características. Por ejemplo, la mediana de `price` es **$35,000** para `None reported` y **$18,000** para `At least 1 accident or damage reported`; esto respalda conservar la distinción de accidente como predictor, no interpretarla como efecto causal. En `clean_title`, la comparación entre valores observados sigue siendo imposible porque solo aparece `Yes`.

### Texto de `engine` y `transmission`

Estos campos son cadenas descriptivas con patrones parcialmente estructurados. Se inspeccionan sus valores más frecuentes; la extracción de atributos se describe aquí; su implementación se realiza en la sección 6 mediante `features.py`.

In [ ]:
for column in ["engine", "transmission"]:
    counts = df[column].value_counts(dropna=False).head(15)
    frequency_table = counts.rename_axis("valor observado").to_frame("frecuencia")
    frequency_table["porcentaje del dataset"] = frequency_table["frecuencia"] / len(df) * 100
    frequency_table["porcentaje del dataset"] = frequency_table["porcentaje del dataset"].map("{:.2f}%".format)
    print(f"Valores más frecuentes de {column}:")
    display(frequency_table)

Los ejemplos de `engine` muestran patrones como `355.0HP 5.3L 8 Cylinder Engine Gasoline Fuel`; potencialmente describen **potencia (HP)**, **cilindrada (L)**, **número de cilindros** y **tipo de combustible/motor**. En `transmission`, valores como `8-Speed A/T` o `6-Speed M/T` sugieren **número de velocidades** y **tipo de transmisión** (automática/manual). Los formatos no son uniformes, por lo que cualquier extracción futura requeriría reglas, categorías no reconocidas y validación; aquí solo se describen, no se extraen.

### Correlación entre variables numéricas

La matriz se limita a `model_year`, `milage` y `price`; `id` se excluye por ser un identificador sin significado cuantitativo para la relación entre características del vehículo. La correlación de Pearson resume asociación lineal y no descarta relaciones no lineales.

In [ ]:
correlation_columns = ["model_year", "milage", "price"]
correlation_matrix = df[correlation_columns].corr(method="pearson")
display(correlation_matrix.round(3))

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(
    correlation_matrix,
    annot=True,
    fmt=".3f",
    cmap="vlag",
    center=0,
    square=True,
    linewidths=0.5,
    ax=ax,
)
ax.set_title("Correlación de Pearson")
plt.tight_layout()
plt.show()

`model_year` y `milage` presentan correlación inversa de **−0.670**: los autos más nuevos tienden a tener menos kilometraje, pero la relación no es redundancia perfecta. Sus correlaciones con `price` son **0.232** y **−0.283**, respectivamente, lo que indica asociaciones lineales moderadas o débiles y deja espacio para efectos no lineales e interacciones. Se justifica conservar ambas para el modelado y excluir `id`; cualquier transformación debe evaluarse con validación, no decidirse solo por esta matriz.

### Conclusiones y decisiones recomendadas

Las siguientes decisiones son recomendaciones para la fase de preprocesamiento, no transformaciones ejecutadas aquí. Deben ajustarse usando únicamente el conjunto de entrenamiento de cada partición para evitar fuga de información.

- `fuel_type` (2.70% de nulos): conservar como categórica; representar los faltantes como `Desconocido` o con imputación categórica más indicador. Revisar y normalizar niveles raros o caracteres no estándar antes de agrupar categorías.
- `accident` (1.30% de nulos): conservar; distinguir el faltante como `Desconocido` y mantener `None reported` separado, pues el precio medio difiere entre filas con y sin nulo.
- `clean_title` (11.36% de nulos): conservar la señal de ausencia como categoría/indicador; entre valores no nulos solo aparece `Yes`, por lo que la columna observada no aporta variación y podría descartarse después de crear el indicador de faltante.
- `id`: descartar como predictor; es un identificador y puede inducir asociaciones espurias.
- `model_year` y `milage`: conservar ambas. Revisar límites y consistencia, y considerar derivar antigüedad del vehículo; no eliminar una por su correlación de −0.670.
- Valores atípicos en `model_year`, `milage` y `price`: inspeccionar plausibilidad y errores, pero no eliminarlos solo por la regla IQR. Comparar modelos robustos y, si hay evidencia, evaluar tratamiento dentro de cada partición.
- `engine`: en una fase posterior evaluar extraer potencia en HP, cilindrada en litros, número de cilindros y tipo de combustible/motor; validar cobertura y formatos no reconocidos.
- `transmission`: normalizar y evaluar derivar tipo (automática/manual u otra) y número de velocidades; conservar una categoría de respaldo para formatos no interpretables.
- Categóricas de alta cardinalidad (`model`, colores, `brand`, `transmission`): comparar codificación apropiada y agrupación de niveles infrecuentes, aprendiendo reglas solo con entrenamiento.
- `price`: probar `log1p(price)` en validación, dado el sesgo 20.27 frente a 0.108 en escala logarítmica. Invertir la transformación al reportar errores en dólares y revisar el sesgo introducido al volver a la escala original.

El dataset original no se altera: todas las tablas y gráficos se calculan sobre `df` en memoria y no hay entrenamiento ni escritura de datos.

## 4. Datos faltantes y estrategia

Primero cuantificamos los nulos, sin eliminar filas de forma automática. Eliminar una fila por cualquier predictor ausente puede descartar muchas observaciones (y sesgar la muestra si la ausencia no es aleatoria). En este conjunto, `fuel_type`, `accident` y `clean_title` son campos categóricos: conservaremos las observaciones e imputaremos una categoría explícita `Desconocido` dentro del pipeline. Así se distingue la falta de registro de una categoría observada y se evita calcular valores usando el test. Los nulos de `engine` o `transmission` también son admisibles: las extracciones quedan en NaN y se imputan en el paso numérico o categórico correspondiente.

In [ ]:
missing = df.isna().sum().rename("nulos")
missing_summary = pd.DataFrame(
    {
        "nulos": missing,
        "porcentaje": (missing / len(df) * 100).round(2),
    }
).sort_values("nulos", ascending=False)
missing_summary

## 5. Partición train/test

Usamos una partición aleatoria 80/20 con semilla fija para reservar una evaluación final reproducible y mantener suficientes filas en train para aprender imputación y categorías. Separamos `price` como objetivo y descartamos `id` antes del split porque es un identificador de fila, no una característica del vehículo. El split aleatorio no revela por sí mismo el objetivo ni usa `id`; sin VIN o identificador de vehículo no podemos hacer una partición agrupada. Registros de vehículos repetidos o muy similares podrían quedar en ambos lados y hacer la evaluación optimista; revisaremos duplicados exactos de predictores, aunque no prueban que sea el mismo vehículo.

In [ ]:
X = df.drop(columns=["price", "id"])
y = df["price"].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
)

print(f"X_train: {X_train.shape} | X_test: {X_test.shape}")
print(f"y_train: {y_train.shape} | y_test: {y_test.shape}")
print(f"Duplicados exactos de predictores (antes del split): {X.duplicated().sum():,}")

## 6. Extracción y preprocesamiento

La extracción de HP, litros, cilindros y velocidades es determinista y por fila. Encadenamos las funciones del módulo como `FunctionTransformer` y después el `ColumnTransformer`; por tanto, el objeto completo se puede guardar con `joblib` y reutilizar en inferencia. Las numéricas usan mediana. Para `fuel_type`, tipo de transmisión, accidente y título limpio usamos una categoría explícita de desconocido y one-hot; `brand`, `model` y colores usan `OrdinalEncoder` con código `-1` para categorías no vistas. Esta decisión controla el tamaño que produciría un one-hot de alta cardinalidad; la codificación ordinal puede ser poco apropiada para modelos lineales. El escalado estándar es útil para modelos lineales y basados en distancias, pero puede omitirse si el estimador final es un árbol.

Este `preprocessing_pipeline` es independiente y se usa para inspeccionar transformaciones y verificar fuga: se ajusta solo con `X_train`. El pipeline final de la sección 10 es otro objeto, independiente, que se vuelve a ajustar dentro de cada fold de la validación cruzada.

In [ ]:
preprocessing_pipeline = Pipeline(
    steps=[
        ("engine_features", FunctionTransformer(extract_engine_features, validate=False)),
        (
            "transmission_features",
            FunctionTransformer(extract_transmission_features, validate=False),
        ),
        ("preprocessor", build_preprocessor()),
    ]
)

# El pipeline aprende imputadores, categorías y escalas exclusivamente de X_train.
preprocessing_pipeline.fit(X_train)
X_train_processed = preprocessing_pipeline.transform(X_train)
X_test_processed = preprocessing_pipeline.transform(X_test)

print(f"X_train antes/después: {X_train.shape} -> {X_train_processed.shape}")
print(f"X_test antes/después:  {X_test.shape} -> {X_test_processed.shape}")

## 7. Outliers y escala del objetivo

No eliminaremos automáticamente outliers de `price` ni de predictores: autos de lujo, antiguos o con alto kilometraje pueden ser observaciones válidas. El resumen por cuantiles y la regla IQR siguiente cuantifican extremos, pero no deciden borrarlos; cualquier recorte tendría que justificarse con conocimiento del dominio y ajustarse usando solo train. Por ahora mantenemos `price` en su escala original para que las métricas sean interpretables en moneda. `log1p(price)` puede reducir asimetría y ser útil si se optimiza error relativo, pero cambia el objetivo y exige invertir con `expm1` (idealmente con `TransformedTargetRegressor`). Queda como mejora futura (ver sección 13); en esta entrega `price` se mantiene en su escala original.

In [ ]:
price_quantiles = y_train.quantile([0.00, 0.25, 0.50, 0.75, 0.95, 0.99, 1.00])
q1, q3 = y_train.quantile([0.25, 0.75])
iqr = q3 - q1
outlier_count = ((y_train < q1 - 1.5 * iqr) | (y_train > q3 + 1.5 * iqr)).sum()

print("Cuantiles de price en train:")
display(price_quantiles)
print(f"Observaciones train fuera de [Q1 - 1.5*IQR, Q3 + 1.5*IQR]: {outlier_count:,}")
print(f"Asimetría price en train: {y_train.skew():.3f}")
print(f"Asimetría log1p(price) en train: {np.log1p(y_train).skew():.3f}")

## 8. Prevención de fuga de información

- `price` se separa como objetivo y no forma parte de `X`.
- `id` se descarta antes de dividir, por lo que no puede actuar como predictor accidental.
- `preprocessing_pipeline` se ajusta solo con `X_train`; `X_test` solo recibe `transform`. El pipeline final del modelo se ajusta de forma independiente dentro de cada fold de validación cruzada.
- Imputación, categorías de los codificadores y parámetros del escalador viven dentro del pipeline y aprenden durante ese mismo `fit` de train.
- Los predictores disponibles son atributos del vehículo y reportes de condición/título. No hay campos explícitos de venta posterior; antes de desplegar, hay que confirmar que `accident` y `clean_title` estén disponibles en el instante real de predicción.
- También se revisaron duplicados exactos de predictores. Sin VIN no es posible probar ni agrupar vehículos repetidos; el split aleatorio podría ser optimista si el mismo vehículo aparece más de una vez. No se usan precios de test para diseñar las transformaciones.

In [ ]:
assert "price" not in X_train.columns and "price" not in X_test.columns
assert "id" not in X_train.columns and "id" not in X_test.columns
assert list(X.columns) == ["brand", "model", "model_year", "milage", "fuel_type", "engine", "transmission", "ext_col", "int_col", "accident", "clean_title"]
assert X_train.index.intersection(X_test.index).empty

fitted_preprocessor = preprocessing_pipeline.named_steps["preprocessor"]
numeric_imputer = fitted_preprocessor.named_transformers_["numeric"].named_steps["imputer"]
numeric_scaler = fitted_preprocessor.named_transformers_["numeric"].named_steps["scaler"]
ordinal_encoder = fitted_preprocessor.named_transformers_["ordinal"].named_steps["encoder"]
onehot_encoder = fitted_preprocessor.named_transformers_["onehot"].named_steps["encoder"]

assert len(numeric_imputer.statistics_) == 6
assert len(numeric_scaler.mean_) == 6
assert len(ordinal_encoder.categories_) == 4
assert len(onehot_encoder.categories_) == 4
assert X_train_processed.shape[0] == len(X_train)
assert X_test_processed.shape[0] == len(X_test)

print("Controles de fuga superados: objetivo/ID excluidos, índices disjuntos y")
print("transformadores ajustados mediante preprocessing_pipeline.fit(X_train).")

## 9. Modelo base

La competición evalúa con RMSE, que penaliza con mayor intensidad los errores grandes: el resultado se expresa en dólares y un error de 20.000 dólares pesa más que dos errores de 10.000 dólares. Lo acompañaremos con MAE, el error absoluto promedio en dólares, más fácil de interpretar y menos sensible a extremos, y con R², una medida adimensional de mejora respecto a predecir la media (puede ser negativa). Ninguna métrica resume por sí sola todos los aspectos de la calidad del modelo.

Primero calculamos dos referencias sin características: predecir siempre la media o siempre la mediana del precio de entrenamiento. Se evalúan en el test reservado; sirven para comprobar si el modelo aprende señal útil, no como candidatos de despliegue.

In [ ]:
def calcular_metricas(y_real, y_pred):
    return {
        "RMSE ($)": mean_squared_error(y_real, y_pred) ** 0.5,
        "MAE ($)": mean_absolute_error(y_real, y_pred),
        "R²": r2_score(y_real, y_pred),
    }

baseline_rows = []
for strategy in ["mean", "median"]:
    baseline = DummyRegressor(strategy=strategy)
    baseline.fit(X_train, y_train)
    predictions = baseline.predict(X_test)
    baseline_rows.append({"Modelo": f"Dummy ({strategy})", **calcular_metricas(y_test, predictions)})

baseline_results = pd.DataFrame(baseline_rows).set_index("Modelo")
display(baseline_results.round({"RMSE ($)": 0, "MAE ($)": 0, "R²": 4}))

Como es esperable, ambos baselines tienen R² cercano a cero o negativo: una constante no captura las diferencias entre vehículos. La media obtiene menor RMSE y la mediana menor MAE, ilustrando la distinta sensibilidad de ambas métricas a errores grandes. Estos valores son la referencia que el modelo con características debe superar.

## 10. Modelo predictivo y validación cruzada

Elegimos un bosque aleatorio porque puede modelar relaciones no lineales e interacciones entre atributos del vehículo sin exigir una forma funcional previa. Su costo se controla con 40 árboles, profundidad máxima de 20, mínimo de tres observaciones por hoja y ejecución paralela. Se reutiliza `build_preprocessor()` junto con las extracciones deterministas de `engine` y `transmission`, todo dentro del mismo `Pipeline`; aunque el escalado no es necesario para árboles, se conserva la definición común del módulo. Mantenemos `price` sin transformación logarítmica para optimizar y reportar las métricas en dólares directamente.

La validación cruzada de cinco folds se realiza únicamente dentro de `X_train, y_train`. Se presentan RMSE y MAE por fold en dólares, y R² sin unidad, junto con promedio y desviación estándar; `cross_validate` clona y ajusta el pipeline de forma independiente en cada fold.

In [ ]:
pipeline = Pipeline(
    steps=[
        ("engine_features", FunctionTransformer(extract_engine_features, validate=False)),
        ("transmission_features", FunctionTransformer(extract_transmission_features, validate=False)),
        ("preprocessor", build_preprocessor()),
        (
            "model",
            RandomForestRegressor(
                n_estimators=40,
                max_depth=20,
                min_samples_leaf=3,
                max_features=0.7,
                n_jobs=-1,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_scores = cross_validate(
    pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring={
        "rmse": "neg_root_mean_squared_error",
        "mae": "neg_mean_absolute_error",
        "r2": "r2",
    },
    n_jobs=1,
    return_train_score=False,
)

dummy_cv_scores = cross_validate(
    DummyRegressor(strategy="mean"),
    X_train,
    y_train,
    cv=cv,
    scoring={
        "rmse": "neg_root_mean_squared_error",
        "mae": "neg_mean_absolute_error",
        "r2": "r2",
    },
    n_jobs=1,
    return_train_score=False,
)

cv_results = pd.DataFrame(
    {
        "RMSE ($)": -cv_scores["test_rmse"],
        "MAE ($)": -cv_scores["test_mae"],
        "R²": cv_scores["test_r2"],
    },
    index=[f"Fold {fold}" for fold in range(1, 6)],
)
cv_summary = pd.DataFrame({"Media": cv_results.mean(), "Desviación estándar": cv_results.std()})
cv_summary_display = cv_summary.copy()
cv_summary_display.loc[["RMSE ($)", "MAE ($)"]] = cv_summary_display.loc[["RMSE ($)", "MAE ($)"]].round(0)
cv_summary_display.loc["R²"] = cv_summary_display.loc["R²"].round(4)
print("Resultados por fold:")
display(cv_results.round({"RMSE ($)": 0, "MAE ($)": 0, "R²": 4}))
print("Resumen de validación cruzada:")
display(cv_summary_display)


cv_comparison = pd.DataFrame(
    [
        {
            "Modelo": "Dummy (mean)",
            "RMSE ($)": -dummy_cv_scores["test_rmse"].mean(),
            "MAE ($)": -dummy_cv_scores["test_mae"].mean(),
            "R²": dummy_cv_scores["test_r2"].mean(),
        },
        {
            "Modelo": "Random Forest",
            "RMSE ($)": -cv_scores["test_rmse"].mean(),
            "MAE ($)": -cv_scores["test_mae"].mean(),
            "R²": cv_scores["test_r2"].mean(),
        },
    ]
).set_index("Modelo")
print("Comparación de medias en los mismos folds:")
display(cv_comparison.round({"RMSE ($)": 0, "MAE ($)": 0, "R²": 4}))

# Después de la validación, ajustamos el pipeline definitivo con todo el conjunto train.
pipeline.fit(X_train, y_train)
pipeline_feature_count = pipeline[:-1].transform(X_train).shape[1]
assert pipeline.named_steps["model"].n_features_in_ == pipeline_feature_count
print("Pipeline final ajustado con X_train; dimensiones procesadas verificadas y X_test sigue reservado para la evaluación final.")

En la validación cruzada de cinco folds, Random Forest superó al `DummyRegressor(strategy="mean")` evaluado con los mismos folds: RMSE **$75,120 vs. $79,667**, MAE **$20,468 vs. $29,137** y R² **0.1111 vs. −0.0001**. Las métricas por fold y su dispersión permiten juzgar estabilidad interna, pero no sustituyen la evaluación final en el test reservado.

## 11. Evaluación final en test

Predecimos el conjunto reservado una sola vez con el pipeline ya ajustado y comparamos el resultado con ambos baselines. La tabla reporta RMSE y MAE en dólares y R² adimensional. Las diferencias porcentuales impresas indican la reducción del error frente al baseline de media; un valor negativo significa que el modelo empeoró esa métrica.

In [ ]:
test_predictions = pipeline.predict(X_test)
model_test_metrics = calcular_metricas(y_test, test_predictions)
final_results = pd.concat(
    [baseline_results, pd.DataFrame([model_test_metrics], index=["Random Forest"])],
    axis=0,
)
print("Comparación en el test reservado:")
display(final_results.round({"RMSE ($)": 0, "MAE ($)": 0, "R²": 4}))

mean_baseline_metrics = baseline_results.loc["Dummy (mean)"]
for metric in ["RMSE ($)", "MAE ($)"]:
    reduction_pct = (1 - model_test_metrics[metric] / mean_baseline_metrics[metric]) * 100
    direction = "reduce" if reduction_pct >= 0 else "aumenta"
    print(f"Frente a Dummy (mean), el modelo {direction} {metric} en {abs(reduction_pct):.2f}%.")
r2_gain = model_test_metrics["R²"] - mean_baseline_metrics["R²"]
print(f"Cambio en R² frente a Dummy (mean): {r2_gain:+.4f} puntos.")

En el test reservado, el bosque logra RMSE de 69.171 dólares, frente a 74.573 del baseline de media, una reducción de 7,24%. El MAE baja de 29.087 a 20.317 dólares (30,15%) y R² sube a 0,1396. El modelo aporta señal útil y mejora las tres métricas, aunque explica solo cerca del 14% de la variabilidad del precio: es un punto de partida moderado, no una precisión suficiente para valorar un vehículo individual sin otros controles. La diferencia entre el RMSE de CV (75.120) y el de test (69.171) recuerda que los errores grandes y la composición del split afectan bastante este problema.

## 12. Predicciones, residuos e importancia de variables

El primer gráfico contrasta precio real y predicho; la diagonal representa predicción perfecta. El segundo muestra la distribución de residuos definidos como `real - predicho`: residuos positivos indican subestimación y negativos, sobreestimación. Estos gráficos permiten detectar compresión hacia la media, asimetría y errores extremos que una sola métrica no revela.

In [ ]:
residuals = y_test.to_numpy() - test_predictions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(y_test, test_predictions, alpha=0.18, s=10, color="#287271")
plot_min = min(y_test.min(), test_predictions.min())
plot_max = max(y_test.max(), test_predictions.max())
axes[0].plot([plot_min, plot_max], [plot_min, plot_max], linestyle="--", color="#b23a48")
axes[0].set(
    title="Precio real vs. predicho",
    xlabel="Precio real (dólares)",
    ylabel="Precio predicho (dólares)",
)
axes[1].hist(residuals, bins=50, color="#e09f3e", edgecolor="white")
axes[1].axvline(0, linestyle="--", color="#b23a48")
axes[1].set(
    title="Distribución de residuos",
    xlabel="Residuo: real - predicho (dólares)",
    ylabel="Frecuencia",
)
fig.tight_layout()
plt.show()
print(f"Residuo medio: {residuals.mean():,.0f} dólares")
print(f"Percentiles 5 y 95 del residuo: {np.percentile(residuals, [5, 95]).round(0)} dólares")

La nube real-predicho debe leerse frente a la diagonal: alejarse de ella representa mayor error. El residuo medio de −1.057 dólares indica una sobreestimación promedio pequeña; sin embargo, entre los percentiles 5 y 95 hay cerca de 79.713 dólares de amplitud. Por tanto, el sesgo global está cerca de cero, pero los errores de casos individuales siguen siendo amplios, especialmente importante para precios extremos.

### Importancia de variables

El bosque permite ordenar las variables por importancia basada en reducción de impureza. La extracción de nombres viene del `ColumnTransformer`, así que aparecen los nombres de los atributos ya procesados. Esta importancia es descriptiva, no causal; con categorías ordinales y variables correlacionadas, puede repartir o sesgar la contribución. No se calcula sobre el test.

In [ ]:
feature_names = pipeline.named_steps["preprocessor"].get_feature_names_out()
feature_importance = pd.DataFrame(
    {
        "Variable transformada": feature_names,
        "Importancia": pipeline.named_steps["model"].feature_importances_,
    }
).sort_values("Importancia", ascending=False)
display(feature_importance.head(15).reset_index(drop=True).round({"Importancia": 4}))

El kilometraje es la variable dominante (importancia 0,371), seguido de año del modelo, modelo del vehículo, color exterior y potencia. Es una lectura coherente con la depreciación y las diferencias entre versiones, pero no implica causalidad. `model` y los colores usan códigos ordinales para controlar cardinalidad; el bosque puede aprovechar cortes de esos códigos, aunque su orden artificial limita la interpretación y podría mejorarse con CatBoost/LightGBM o una codificación categórica más adecuada.

## 13. Interpretación y siguientes versiones

El modelo mejora al baseline de media en test en 5.402 dólares de RMSE (7,24%) y 8.770 dólares de MAE (30,15%); el aumento de R² es 0,1396. La mejora es consistente en las tres métricas, pero el RMSE de 69.171 dólares y el R² de 0,1396 muestran que aún quedan errores grandes y mucha variación sin explicar. La magnitud del RMSE es razonable como primera referencia frente al baseline, no como resultado final competitivo: RMSE enfatiza vehículos extremos, mientras el MAE describe mejor el error típico.

Las dificultades principales son precios muy heterogéneos, faltantes y textos de `engine`/`transmission` con formatos distintos, además de categorías de alta cardinalidad. El split aleatorio puede ser optimista si existen vehículos o anuncios repetidos. Próximos pasos: probar boosting (CatBoost/LightGBM si está disponible), búsqueda acotada de hiperparámetros solo dentro de validación cruzada, comparar `log1p(price)` mediante `TransformedTargetRegressor` reportando siempre métricas en escala original, revisar errores por rangos de precio y marcas, y usar particiones por grupo o tiempo si se dispone de identificadores/fechas que representen el uso real.

## 14. Guardado y prueba de inferencia

Guardamos con `joblib` el pipeline completo, no solo el estimador: así quedan disponibles las extracciones y transformaciones necesarias en producción. El archivo se guarda en `FASE_DIR / "modelo.joblib"`, junto a `features.py`. Después se carga y se verifica que prediga igual sobre el test; finalmente se hacen predicciones sobre tres registros crudos con las columnas originales, sin `id` ni `price`.

In [ ]:
model_path = FASE_DIR / "modelo.joblib"
joblib.dump(pipeline, model_path, compress=3)
loaded_pipeline = joblib.load(model_path)
loaded_test_predictions = loaded_pipeline.predict(X_test)
predictions_match = np.allclose(
    test_predictions, loaded_test_predictions, rtol=1e-12, atol=1e-8
)
assert predictions_match, "Las predicciones del pipeline cargado no coinciden."
print(f"Pipeline guardado: {model_path.resolve()}")
model_size_mb = model_path.stat().st_size / (1024 ** 2)
print(f"Tamaño del archivo: {model_size_mb:.2f} MB")
assert model_size_mb < 50, "El modelo guardado debe pesar menos de 50 MB."
print(f"Predicciones originales y cargadas coinciden: {predictions_match}")

raw_examples = pd.DataFrame(
    [
        {
            "brand": "BMW",
            "model": "3 Series 330i xDrive",
            "model_year": 2020,
            "milage": 35000,
            "fuel_type": "Gasoline",
            "engine": "255.0HP 2.0L 4 Cylinder Engine",
            "transmission": "8-Speed Automatic",
            "ext_col": "Black",
            "int_col": "Black",
            "accident": "None reported",
            "clean_title": "Yes",
        },
        {
            "brand": "Toyota",
            "model": "Camry SE",
            "model_year": 2018,
            "milage": 65000,
            "fuel_type": "Gasoline",
            "engine": "203.0HP 2.5L 4 Cylinder Engine",
            "transmission": "8-Speed Automatic",
            "ext_col": "Silver",
            "int_col": "Gray",
            "accident": "None reported",
            "clean_title": "Yes",
        },
        {
            "brand": "Ford",
            "model": "F-150 XLT",
            "model_year": 2021,
            "milage": 42000,
            "fuel_type": "Gasoline",
            "engine": "400.0HP 5.0L 8 Cylinder Engine",
            "transmission": "10-Speed Automatic",
            "ext_col": "Oxford White",
            "int_col": "Black",
            "accident": "At least 1 accident or damage",
            "clean_title": "Yes",
        },
    ],
    columns=X.columns,
)
raw_predictions = loaded_pipeline.predict(raw_examples)
example_results = raw_examples.copy()
example_results["price_predicho"] = raw_predictions
print("Predicciones sobre tres filas crudas (sin id ni price):")
display(example_results[["brand", "model", "model_year", "milage", "price_predicho"]].round({"price_predicho": 0}))

La serialización se completó y el pipeline cargado reproduce las predicciones del test (`np.allclose` devuelve `True`). El archivo contiene el preprocesador y el bosque; las tres filas crudas se procesan directamente, sin construir variables derivadas a mano. Las predicciones de ejemplo son aproximadamente 38.941, 31.972 y 46.955 dólares, respectivamente.